## tl;dr

- 라이브 브랜드 탭 백테스트는 **17/18개 검증을 통과**했습니다.
- **High 2건**: SKU 집중도 분모 오류와 다년 월별 비교의 달력월 합산 왜곡이 확인됐습니다.
- SKU 집중도 오류는 **60/71개 브랜드**에 영향을 줍니다.
- 다년 월 합산 때문에 최고월이 달라지는 브랜드는 **7개**입니다.

## Context & Methods

브랜드 탭의 브랜드 순위, 매출·수량·SKU·국가 KPI, MoM/YoY 성장, SKU 집중도, 국가/제품군 분포, 월별 비교를 최신 분석 API 결과에서 독립 재계산했습니다.

### Key Assumptions

- 브랜드 탭의 정본은 `countrySkuSummary`, 월별 정본은 `countrySkuMonthly`입니다.
- 성장률은 `monthCoverage.status == complete`인 월만 비교합니다.
- SKU 집중도는 브랜드 전체 매출을 분모로 해석합니다.
- 다년 시즌성은 완료월만 사용하고 달력월별 관측 연도 수로 평균해야 비교 가능합니다.

## Data

In [1]:
from pathlib import Path
import json
import pandas as pd

result_path = Path('artifacts/brand_backtest_20260715.json')
result = json.loads(result_path.read_text(encoding='utf-8'))
source_summary = pd.DataFrame([{
    '기간': f"{result['source']['analysisOptions']['start_date']} ~ {result['source']['analysisOptions']['end_date']}",
    '브랜드': result['source']['brandCount'],
    '요약행': result['source']['summaryRows'],
    '월별행': result['source']['monthlyRows'],
    '전체월': result['source']['monthCount'],
    '완료월': result['source']['completeMonthCount'],
}])
source_summary

,기간,브랜드,요약행,월별행,전체월,완료월
0,2024-07-14 ~ 2026-07-14,71,28246,112952,25,23


## Results

In [2]:
checks = pd.DataFrame([{
    '검증': item['id'],
    '결과': 'PASS' if item['passed'] else 'FAIL',
} for item in result['checks']])
checks

,검증,결과
0,payload.status,PASS
1,payload.schema,PASS
2,source.non_empty,PASS
3,source.total_amount_reconciliation,PASS
4,source.total_qty_reconciliation,PASS
5,source.summary_grain_unique,PASS
6,source.monthly_grain_unique,PASS
7,brand.summary_monthly_reconciliation,PASS
8,month.coverage_reconciliation,PASS
9,growth.complete_month_only,PASS


In [3]:
findings = pd.DataFrame([{
    '심각도': item['severity'].upper(),
    '이슈': item['title'],
    '영향': item['impact'],
    '조치': item['recommendation'],
} for item in result['findings']])
findings

,심각도,이슈,영향,조치
0,HIGH,월별 추이 비교가 여러 연도의 같은 달을 합산합니다.,"장기 분석에서는 서로 다른 연도의 같은 달이 합쳐지고, 월별 관측 횟수와 부분월 포...","의도가 시즌성이라면 완료월만 사용해 달력월별 관측 연도 수로 나눈 평균을 표시하고,..."
1,HIGH,SKU 집중도가 브랜드 전체가 아니라 상위 5개 SKU 합계를 분모로 계산됩니다.,상위 5개 막대의 표시 비중이 항상 약 100%로 합산되어 실제 브랜드 내 집중도를...,분모를 selectedBrand.amount 또는 선택 브랜드의 전체 SKU 매출 ...
2,MEDIUM,브랜드 순위/KPI와 성장률의 반영 월 범위가 다릅니다.,순위·점유율·SKU/국가 분포에는 부분월이 포함되지만 MoM/YoY 성장률에서는 제...,"화면에 범위 차이를 명시하거나, 브랜드 요약에도 완료월 필터를 적용할지 제품 기준을..."
3,MEDIUM,국가 또는 카테고리 매핑이 완전하지 않습니다.,국가 분포와 제품군 구성비에 '미상/미분류'가 포함되어 Top 5와 점유율 해석이 ...,매핑률을 브랜드 탭에 함께 표시하고 미상/미분류 비중 임계치를 자동 점검하세요.


In [4]:
sku_issue = next(item for item in result['findings'] if item['id'] == 'sku_concentration_top5_denominator')
sku_examples = pd.DataFrame(sku_issue['evidence']['worstExamples'])
sku_examples[['brand', 'skuCount', 'trueTopFiveSharePct', 'uiTopFiveSharePct', 'trueTopOneSharePct', 'uiTopOneSharePct', 'topOneOverstatementPp']]

,brand,skuCount,trueTopFiveSharePct,uiTopFiveSharePct,trueTopOneSharePct,uiTopOneSharePct,topOneOverstatementPp
0,헤이미쉬,92,38.868954,100,17.427078,44.835470,27.408392
1,라카,52,21.242065,100,6.142355,28.916000,22.773644
2,넘버즈인,24,44.062445,100,17.235021,39.114989,21.879969
3,스튜디오 17,31,34.191176,100,10.798564,31.582896,20.784332
4,조선미녀,79,53.367728,100,21.998736,41.221047,19.222311
5,퓌,66,30.250077,100,8.296692,27.427011,19.130319
6,코스알엑스,118,41.173696,100,12.850618,31.210747,18.360129
7,메디큐브,96,43.251413,100,13.230762,30.590359,17.359596
8,마리엔메이,72,26.172311,100,6.067437,23.182656,17.115219
9,EDGE U,60,19.995717,100,4.205177,21.030389,16.825212


In [5]:
season_issue = next(item for item in result['findings'] if item['id'] == 'seasonality_year_aggregation')
peak_examples = pd.DataFrame(season_issue['evidence']['samples'])
peak_examples

,brand,uiPeakMonth,uiPeakAmount,normalizedPeakMonth,normalizedPeakAverageAmount
0,라운드랩,5,1126863.17,7,607470.760
1,믹순,7,204815.43,3,73008.525
2,빌리프,7,800.00,12,70.000
3,아임프롬,7,16645.07,6,5534.915
4,쿤달,7,6174.97,6,3073.035
5,툴리프,7,907.86,6,261.015
6,프랭클리,7,2421.36,6,195.350


## Takeaways

1. SKU 집중도 분모를 브랜드 전체 SKU 매출 합계로 수정해야 합니다.
2. 월별 비교는 `year_month` 추이와 달력월 시즌성을 분리해야 합니다. 시즌성은 완료월·연평균 기준이 안전합니다.
3. 부분월 포함 범위와 성장률 완료월 범위를 화면에 명확히 표시해야 합니다.
4. 미분류 카테고리 매출 비중을 상시 품질 지표로 모니터링하는 것이 좋습니다.